# Complexity-03e — Barrières d'exposants et word RAM

**Navigation** : [Index](README.md) | [Version Python](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) | [03 — Hiérarchie temporelle](Complexity-03-HierarchieTemporelle-Python.ipynb) | [03f — Réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)

**Objectifs** : lire des exposants rationnels, exécuter les neuf instructions d'un word RAM et confronter un témoin au décompte manuel ; mesurer un contrôle 3SUM quadratique. **Prérequis** : C#, complexité et complément à deux. **Durée estimée** : 60 minutes.

Ce carnet lit une [formalisation publiée](https://github.com/anthropics/formal-math/tree/main/3sum-apsp), sans exécuter les programmes des auteurs ni `lake build`. Les cinq claims sont des existences de programmes et leurs constantes ne sont pas optimisées. Le contrôle quadratique ne mesure pas le gain annoncé.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Diagnostics;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;
const int SEED=20261028;
// Graine fixe : batterie 3SUM et mesures de temps du carnet partent toutes de cette meme source.
Console.WriteLine($"{System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; BigInteger ; graine {SEED}");

.NET 10.0.15 ; BigInteger ; graine 20261028


### Ce que montre cette sortie

La sortie ne montre pas un résultat mais l'**environnement** qui rendra comparables tous les relevés du carnet : la version du runtime, la présence de `BigInteger` pour l'arithmétique non bornée, et la **graine imposée** `20261028` dont partent la batterie de tirage et les chronos.

Trois lectures.

- **La graine est l'instrument de reproductibilité.** Le carnet mesure des temps, et un temps n'est reproductible que si le tirage aléatoire l'est aussi. La graine est déclarée une fois, en tête, et réutilisée telle quelle : deux exécutions du carnet sur la même machine produisent la même batterie.
- **Les deux langages ne partagent pas le générateur.** Le carnet le dit ailleurs : `System.Random` n'est pas le générateur de Python. Le jumeau Python ne rejouera donc pas les mêmes tirages, seulement la même loi — ce qui suffit pour une vérification, pas pour une comparaison à l'identique.
- **La version du runtime est une donnée, pas un décor.** Elle conditionne les performances mesurées plus bas, et donc la lecture qu'on peut faire des temps affichés.

Ce que cette sortie **ne** prouve pas : rien sur les bornes citées. Elle cadre le milieu dans lequel les relevés seront pris.
## Cinq claims cités

| Énoncé | Problème | Borne citée |
|---|---|---|
| Theorem 19 | Exact Triangle | $O(n^{2,9983})$ |
| Theorem 22 | 3SUM | $O(n^{1,9992})$ |
| Theorem 22 | produit (min,+) | $O(n^{2,99942})$ |
| Theorem 22 | APSP | $O(n^{2,99942})$ |
| Corollary 39 | k-Clique de poids nul | $O(n^{k-0,0017\lfloor k/3\rfloor})$ |

Le fichier `EndStatement.lean` énonce ces propositions. Le papier donne deux bornes pour le théorème 22 : les claims reprennent les plus petites. Le BigO rationnel élève les deux membres à la puissance du dénominateur.

Pour Theorem 22, le papier présente deux algorithmes : les bornes numériques citées sont 1,99923 et 1,9992 pour 3SUM, puis 2,99949 et 2,99942 pour le produit (min,+) et APSP. Les claims formels prennent les seconds nombres, plus petits. Il s'agit de bornes théoriques citées, pas de chronos exécutés ici.

In [2]:
// Deux barrieres d'exposants rationnels r > 2 : elles separent T(n)^r (bas) des algorithmes en n^(2+epsilon) (haut).
foreach(var (numerateur,denominateur) in new[]{(2499,1250),(149971,50000)})
 Console.WriteLine($"r = {numerateur}/{denominateur} : T(n)^{denominateur} <= K * n^{numerateur}, n >= 2");

r = 2499/1250 : T(n)^1250 <= K * n^2499, n >= 2


r = 149971/50000 : T(n)^50000 <= K * n^149971, n >= 2


Les deux fractions sont exactes, et non des arrondis introduits ici. Le projet décrit les preuves mais ce carnet ne les recharge pas.

### Exercice 1 — quotient de puissances

Retourner $n^{\Delta}$ pour un exposant positif. Tester deux couples non triviaux ; une fois la fonction complétée, calculer les écarts des deux algorithmes pour 3SUM et pour (min,+)/APSP, puis leurs facteurs à $10^6$ et $10^{12}$.

Les deux bornes affichées sont les **énoncés** des barrières, pas des relevés : `T(n)^1250 <= K * n^2499` et `T(n)^50000 <= K * n^149971`, chacune pour `n >= 2`. La sortie ne fait que transcrire deux couples d'entiers ; l'exposant de la barrière est la racine de la fraction, et l'élévation à la puissance du dénominateur sert à supprimer la fraction elle-même.

Ce que cela achète : un exposant rationnel devient une inégalité entre puissances entières, dans laquelle les deux membres se comparent sans fraction, et où le dénominateur joue le rôle de multiplicateur de la hauteur. Ce que cela ne dit pas : rien sur la constante `K`, ni sur la taille minimale de `n` à partir de laquelle la barrière mord, ni sur l'algorithme qui la franchit. Une barrière d'exposants est une **restriction** : elle interdit un certain algorithme, elle ne prouve pas qu'un autre existe.

Autrement dit, la sortie atteste que les fractions ne sont pas des arrondis : elle n'avance en rien sur la difficulté réelle du problème cité.

In [3]:
// Le facteur n^delta dit combien de fois plus lent qu'un algorithme n^2 on peut etre tout en restant sous la barriere.
double? Facteur(double n,double delta) { // TODO : calculer n puissance delta
 return null;
}
var facteur=Facteur(16,0.5);
if(facteur is null) Console.WriteLine("Exercice 1 à compléter : quotient des puissances.");
else { var autre=Facteur(1000,1.0/3); if(!double.IsFinite(facteur.Value)||Math.Abs(facteur.Value-4)>1e-9||autre is null||!double.IsFinite(autre.Value)||Math.Abs(autre.Value-10)>1e-9) throw new Exception("Facteur incorrect"); Console.WriteLine("Deux fixtures du facteur validées."); foreach(var (probleme,delta) in new[]{("3SUM",0.00003),("min-plus",0.00007),("APSP",0.00007)}) { var f6=Facteur(1e6,delta); var f12=Facteur(1e12,delta); if(f6 is null || f12 is null || !double.IsFinite(f6.Value) || !double.IsFinite(f12.Value) || Math.Abs(f6.Value-Math.Pow(1e6,delta))>1e-9 || Math.Abs(f12.Value-Math.Pow(1e12,delta))>1e-9) throw new Exception("Facteurs incorrects"); Console.WriteLine($"{probleme} : écart {delta:F5}, facteur 10^6={f6:F6}, 10^12={f12:F6}"); } }

Exercice 1 à compléter : quotient des puissances.


### Ce que montre cette sortie

La sortie ne montre pas un calcul mais un **squelette** : elle dit exactement ce qui manque — la fonction qui doit rendre le quotient des puissances — et elle le dit sans lever d'erreur, ce qui permet au carnet de s'exécuter de bout en bout.

Ce que ce choix de squelette permet de lire :

- **Le contrat est dans le test, pas seulement dans l'énoncé.** La fonction est appelée sur deux couples de valeurs, et les résultats attendus sont inscrits dans la condition de contrôle. Un lecteur peut donc écrire la fonction et savoir immédiatement si elle est juste, sans avoir à interpréter l'énoncé.
- **Le même squelette sert de garde pour la suite.** Les trois écarts annoncés — `3SUM`, `min-plus`, `APSP` — passent par le même appel, à deux tailles différentes. Un seul geste à écrire, trois relevés à produire.
- **L'absence de valeur est un marqueur, pas un résultat.** Tant que la fonction renvoie `null`, la branche de vérification n'est jamais atteinte et rien n'est contrôlé. L'échec du carnet est signalé, jamais masqué.

Ce que cette sortie **ne** prouve pas : elle ne donne aucun facteur. Le rapport entre le coût d'un algorithme et le carré de la taille de l'entrée reste à calculer — c'est précisément l'objet de l'exercice.
## Machine à neuf instructions

`one`, `add`, `sub`, `mul`, `load`, `store`, `bltz`, `accept`, `reject` coûtent un pas chacun, arrêt compris. La mémoire, initialement nulle hors entrée, est indexée par des entiers signés ; les mots se calculent modulo $2^W$. `load` et `store` interprètent l'adresse indirecte en complément à deux. Une position hors programme rejette. Il n'y a ni division, ni décalage, ni opération bit à bit, et la seule constante est 1.

La machine ci-dessous est un interpréteur de lecture et non une preuve des claims. Le budget borne son exécution.

In [4]:
record Instruction(string Op, int A=0, int B=0, int C=0);
record Resultat(bool Verdict, Dictionary<BigInteger,BigInteger> Memoire, int Pas);
// Machine a mots de w bits : NonSigne ramene toute valeur dans [0, 2^w), Signe relit le meme mot en complement a deux.
BigInteger NonSigne(BigInteger v,int w) { var m=BigInteger.One<<w; return ((v%m)+m)%m; }
BigInteger Signe(BigInteger v,int w) { v=NonSigne(v,w); var m=BigInteger.One<<w; return v >= (m>>1) ? v-m : v; }
Dictionary<BigInteger,BigInteger> Charger(int w,params int[] mots) { var d=new Dictionary<BigInteger,BigInteger>(); for(int i=0;i<mots.Length;i++) d[i]=NonSigne(mots[i],w); return d; }
// Exec : chaque pas compte ; budget epuise ne rend AUCUN verdict (null), ce n'est pas un reject.
Resultat Exec(Instruction[] p,Dictionary<BigInteger,BigInteger> mem0,int w,int budget) {
 var mem=new Dictionary<BigInteger,BigInteger>(mem0); int pc=0; BigInteger Lire(BigInteger a)=>mem.GetValueOrDefault(a,BigInteger.Zero);
 for(int pas=1;pas<=budget;pas++) { var ins=pc>=0 && pc<p.Length ? p[pc] : new Instruction("reject");
  switch(ins.Op) {
   case "one": mem[ins.A]=NonSigne(1,w); pc++; break;
   case "add": mem[ins.A]=NonSigne(Lire(ins.B)+Lire(ins.C),w); pc++; break;
   case "sub": mem[ins.A]=NonSigne(Lire(ins.B)-Lire(ins.C),w); pc++; break;
   case "mul": mem[ins.A]=NonSigne(Lire(ins.B)*Lire(ins.C),w); pc++; break;
   case "load": mem[ins.A]=Lire(Signe(Lire(ins.B),w)); pc++; break;
   case "store": mem[Signe(Lire(ins.A),w)]=Lire(ins.B); pc++; break;
   case "bltz": pc=Signe(Lire(ins.A),w)<0 ? ins.B : pc+1; break;
   case "accept": return new(true,mem,pas);
   case "reject": return new(false,mem,pas);
   default: throw new ArgumentException("Instruction inconnue");
  }
 } return null;
}
Console.WriteLine("Interpréteur chargé : neuf instructions ; budget et arrêt comptés.");

Interpréteur chargé : neuf instructions ; budget et arrêt comptés.


### Ce que montre cette sortie

La sortie confirme un **démarrage**, pas un calcul : l'interpréteur est chargé, ses neuf instructions sont énumérées, et deux conventions sont annoncées — le budget d'exécution est compté, et l'arrêt lui-même coûte un pas.

Ce qui est établi par cette phrase courte :

- **Le budget n'est pas un refus.** L'interpréteur renvoie `null` quand le budget est épuisé : aucun verdict n'est rendu. La distinction entre « le programme a rejeté » et « le calcul n'a pas abouti » est le point sur lequel repose toute lecture des témoins, et la sortie l'énonce avant même le premier test.
- **L'arrêt est compté.** Le nombre de pas inclut l'instruction d'arrêt, ce qui explique qu'un chemin qui rejette tôt soit plus court qu'un chemin qui mène à `accept`.

Ce que cette sortie **ne** prouve pas : elle ne dit rien de la correction de la machine. Un interpréteur qui s'exécute n'est pas un interpréteur correct ; le carnet le rappellera, et la distinction est reprise sur les témoins.
### Débordement et adressage

Tester explicitement la réduction modulo et l'adresse indirecte négative, en dehors du témoin 3SUM où les sommes restent dans la plage signée. Le débordement ne doit pas être confondu avec la somme entière.

In [5]:
// Temoin d'arithmetique modulaire : sur 4 bits, 7+2 se lit -7 en complement a deux et l'adresse -1 est adressable.
var programmeTest=new[]{new Instruction("add",3,1,2),new Instruction("store",4,3),new Instruction("load",5,4),new Instruction("mul",6,1,2),new Instruction("accept")};
var memTest=Charger(4,0,7,2,0,-1); var test=Exec(programmeTest,memTest,4,10);
if(test==null || Signe(test.Memoire[3],4)!=-7 || test.Memoire[-1]!=9 || test.Memoire[5]!=9 || test.Memoire[6]!=14 || test.Pas!=5) throw new Exception("Erreur débordement/adresse");
Console.WriteLine($"W=4 : 7+2 devient {Signe(test.Memoire[3],4)}, case -1={test.Memoire[-1]}, 7*2 modulo 16={test.Memoire[6]}, {test.Pas} pas.");

W=4 : 7+2 devient -7, case -1=9, 7*2 modulo 16=14, 5 pas.


### Ce que montre cette sortie

La sortie met en scène le débordement et l'adressage indirect dans un seul scénario : sur quatre bits, `7+2` ne vaut pas `9` mais se lit `-7` en complément à deux, l'adresse `-1` est adressable et contient `9`, et `7*2` calculé modulo `16` vaut `14`. Le tout tient en cinq pas.

Quatre enseignements, dont trois contre-intuitifs.

- **L'addition ne déborde pas, elle se replie.** La machine travaille modulo la puissance de deux ; `7+2` ne devient pas un grand nombre, il redevient `-7`. C'est précisément ce dont le branchement se sert : il compare le caractère négatif de la valeur, ce qui n'a de sens que parce que la représentation est bouclée.
- **Une adresse négative est une adresse.** L'écriture puis la relecture à l'adresse `-1` fonctionne, et la valeur relue est bien celle qui y avait été posée. La mémoire est indexée par des entiers signés : le branchement peut donc viser une case située avant l'entrée.
- **La réduction s'applique à la multiplication aussi.** `7*2` rend `14` ; la règle est la même que pour l'addition, et elle s'applique à tout produit dont la taille dépasse le mot.
- **Le nombre de pas est vérifiable.** Cinq pas pour cinq instructions exécutées jusqu'à l'arrêt : le témoin est minimal, aucune instruction superflue.

Ce que cette sortie **ne** prouve pas : rien sur la correction générale de la machine. Elle teste un cas particulier, choisi précisément parce qu'il déborde.
### Exercice 2 — fabriquer une constante

La machine ne possède que `one`. Construire un programme qui laisse $k$ en cellule 1, par doubles et additions. Un `accept` termine le programme ; les fixtures contrôlent 10 et 13, pas une constante unique.

In [6]:
// Contrat de l'exercice : le programme doit laisser la constante k dans la case 1 et terminer sur accept.
Instruction[] ConstruireConstante(int k) { // TODO : fabriquer k, puis accept
 return null;
}
var construit=ConstruireConstante(10);
if(construit is null) Console.WriteLine("Exercice 2 à compléter : programme de construction.");
else { var a=Exec(construit,new(),16,100); var autre=ConstruireConstante(13); var b=autre is null ? null : Exec(autre,new(),16,100); if(a is null || !a.Verdict || a.Memoire.GetValueOrDefault(1)!=10 || b is null || !b.Verdict || b.Memoire.GetValueOrDefault(1)!=13) throw new Exception("Programme incorrect"); Console.WriteLine("Constantes 10 et 13 validées."); }

Exercice 2 à compléter : programme de construction.


### Ce que montre cette sortie

La sortie est un **squelette** : elle indique que le programme de construction reste à écrire, et elle le fait sans échouer, comme le précédent.

Ce que la lecture du code ajoute :

- **Le contrat est double.** Le programme doit d'une part laisser la constante dans la case désignée, d'autre part se conclure par l'instruction d'arrêt. Un programme qui produit la bonne valeur mais ne s'arrête pas ne satisfait pas le contrat — et l'interpréteur le rappellerait par un retour sans verdict plutôt que par une erreur.
- **Les valeurs de contrôle sont inscrites dans le test.** Elles sont écrites en toutes lettres dans la condition de vérification, avec un second cas à contrôler : un programme qui réussit sur un cas peut échouer sur l'autre.
- **La construction se fait sans constante.** La machine ne dispose que de l'instruction qui écrit un ; la constante demandée doit donc être fabriquée par additions et doublements. C'est la raison d'être de l'exercice : écrire un programme, pas un calcul.

Ce que cette sortie **ne** prouve pas : elle ne dit rien de la faisabilité du programme à écrire. La faisabilité est l'objet de l'exercice, pas de sa sortie.
## Témoin fermé à trois nombres

Le programme additionne les trois mots, fabrique zéro, puis rejette si la somme ou son opposé est négatif. Les chemins attendus sont 7 pas pour zéro, 6 pour une somme négative et 7 pour une somme positive. Ce test suppose l'absence de débordement signé ; la batterie utilise de petits entiers.

In [7]:
// Decideur 3SUM : teste a+b+c=0 par soustractions modulaires et branchements bltz.
Instruction[] P3={new("add",10,1,2),new("add",10,10,3),new("sub",11,10,10),new("sub",12,11,10),new("bltz",10,7),new("bltz",12,7),new("accept"),new("reject")};
foreach(var (a,b,c,attendu,pas) in new[]{(-5,2,3,true,7),(-3,-2,-1,false,6),(1,2,3,false,7)}) { var r=Exec(P3,Charger(16,3,a,b,c),16,100); if(r is null||r.Verdict!=attendu||r.Pas!=pas) throw new Exception("Témoin incorrect"); Console.WriteLine($"[{a},{b},{c}] : {r.Verdict}, {r.Pas} pas (attendu {attendu}, {pas})"); }
// Batterie croisee : 300 tirages, un tiers forces a somme nulle, verdict compare a la somme exacte.
var rng=new Random(SEED); int accords=0;
for(int i=0;i<300;i++) { int[] x=Enumerable.Range(0,3).Select(_=>rng.Next(-20,21)).ToArray(); if(rng.NextDouble()<0.35) { int a=rng.Next(3),b=(a+1+rng.Next(2))%3; x[3-a-b]=-x[a]-x[b]; } var r=Exec(P3,Charger(16,3,x[0],x[1],x[2]),16,100); if(r!=null && r.Verdict==(x.Sum()==0)) accords++; }
Console.WriteLine($"Batterie : {accords}/300 accords, System.Random distinct de Python.");
// Hors programme = reject immediat ; budget epuise = aucun verdict (null), pas un reject.
var debord=Exec(P3,Charger(4,3,7,7,2),4,100); if(debord is null || !debord.Verdict) throw new Exception("Débordement attendu");
var hors=Exec(Array.Empty<Instruction>(),new(),4,1); var borne=Exec(P3,Charger(16,3,0,0,0),16,1); if(hors is null || hors.Verdict || hors.Pas!=1 || borne is not null) throw new Exception("Budget ou rejet hors programme incorrect");
Console.WriteLine("W=4 : 7+7+2 vaut 0 modulo 16 : accept malgré somme entière 16 ; hors-programme rejette en un pas et budget épuisé ne rend aucun verdict.");

[-5,2,3] : True, 7 pas (attendu True, 7)


[-3,-2,-1] : False, 6 pas (attendu False, 6)


[1,2,3] : False, 7 pas (attendu False, 7)


Batterie : 300/300 accords, System.Random distinct de Python.


W=4 : 7+7+2 vaut 0 modulo 16 : accept malgré somme entière 16 ; hors-programme rejette en un pas et budget épuisé ne rend aucun verdict.


Le témoin vérifie trois chemins et une batterie finie, sans établir la correction générale de la machine. Les graines sont fixes mais les générateurs des deux langages diffèrent. À W=4, une somme entière de 16 devient zéro modulo 16 : ce contre-exemple est hors du domaine sans débordement du témoin borné.

Les trois lignes de tête disent la même chose de trois façons : le chemin qui mène à `accept` est le plus long, le chemin qui rejette sur une somme négative est le plus court, et celui qui rejette sur une somme positive fait autant de pas que le premier, parce qu'il doit vérifier l'autre terme avant de conclure. L'écart d'un pas entre un rejet tardif et une acceptation est exactement ce qu'on attend d'un branchement qui saute une instruction.

La batterie ajoute ce que trois cas isolés ne peuvent pas établir : trois cents tirages, dont un tiers forcé à somme nulle, et un accord complet entre le verdict de la machine et la somme calculée exactement. Un tel accord est une **propriété empirique** sur une distribution, pas une preuve : il peut tenir sans que quoi que ce soit soit garanti.

La dernière ligne est plus instructive que les trois cents accords. À quatre bits, `7+7+2` vaut zéro modulo `16`, alors que la somme entière vaut `16` : la machine **retient** donc un triplet dont la somme n'est pas nulle au sens entier. C'est la contre-image exacte du test de débordement de la section précédente, et elle énonce la limite du témoin : sur seize bits, le cas n'est plus visible, parce que le domaine du témoin exclut le débordement signé. Ce que le témoin borné prouve, il le prouve **sous** l'hypothèse qu'il affiche.

Ce que ces sorties **ne** prouvent pas : la correction de la machine pour une taille quelconque, ni au-delà de la batterie. Les graines sont fixes mais les générateurs des deux langages diffèrent : le jumeau Python ne rejouera pas les mêmes tirages.

## Contrôle 3SUM quadratique

Tri et deux pointeurs par ancre, sur des entrées positives sans solution : la boucle parcourt son travail complet. Trois répétitions par taille, médiane, puis pente log-log. Le chronomètre et les constantes dépendent du runtime.

In [8]:
// Reference n^2 attendu : balayage deux pointeurs sur tableau trie, mediane de 3 temps par taille.
bool TroisSomme(int[] xs) { var t=xs.OrderBy(x=>x).ToArray(); for(int i=0;i<t.Length-2;i++) { int g=i+1,d=t.Length-1; long cible=-(long)t[i]; while(g<d) { long s=(long)t[g]+t[d]; if(s==cible)return true; if(s<cible)g++; else d--; } } return false; }
if(!TroisSomme(new[]{-3,1,2}) || TroisSomme(new[]{1,2,3})) throw new Exception("Fixtures 3SUM incorrectes");
var tailles=new[]{1000,2000,4000,8000}; var mesures=new List<(int n,double t)>(); rng=new Random(SEED); TroisSomme(new[]{1,2,3});
foreach(int n in tailles) { int[] x=Enumerable.Range(0,n).Select(_=>rng.Next(1,n*n+1)).ToArray(); var temps=new List<double>(); bool rep=true; for(int j=0;j<3;j++) { var sw=Stopwatch.StartNew(); rep=TroisSomme(x); sw.Stop(); temps.Add(sw.Elapsed.TotalSeconds); } temps.Sort(); mesures.Add((n,temps[1])); Console.WriteLine($"n={n}: {temps[1]*1000:F2} ms, médiane de 3, réponse {rep}"); }
var lx=mesures.Select(t=>Math.Log(t.n)).ToArray(); var ly=mesures.Select(t=>Math.Log(t.t)).ToArray(); double mx=lx.Average(),my=ly.Average(); double pente=lx.Zip(ly,(x,y)=>(x-mx)*(y-my)).Sum()/lx.Sum(x=>(x-mx)*(x-mx));
Console.WriteLine($"Pente log-log : {pente:F3} ; ecart a n^2 a 8000 (8000^(pente-2)) : {Math.Pow(8000,pente-2):F2}");

n=1000: 1.01 ms, médiane de 3, réponse False


n=2000: 3.82 ms, médiane de 3, réponse False


n=4000: 16.04 ms, médiane de 3, réponse False


n=8000: 60.10 ms, médiane de 3, réponse False


Pente log-log : 1.977 ; ecart a n^2 a 8000 (8000^(pente-2)) : 0.81


Le facteur $8000^{p-2}$, où $p$ est la pente mesurée ci-dessus, compare les seules puissances : il vaut 1 quand la croissance est exactement quadratique et mesure donc l'écart à $n^2$. Il ne prédit pas le rapport des temps : les constantes et termes cachés sont inconnus ; aucune implémentation de la borne citée n'est exécutée ici.

### Exercice 3 — comptage des comparaisons

Sur une entrée strictement positive triée, compter les itérations de la boucle à deux pointeurs pour toutes les ancres : c'est un sous-calcul du coût. Puis proposer une traduction explicite d'une itération en instructions `load`, `sub`, `bltz` et déplacements d'indices, et dériver un ordre de grandeur en pas de `exec` pour le décideur tri + deux pointeurs. Justifier aussi le coût du tri et la taille des mots. Comparer aux chronos sans assimiler des itérations à des pas machine : leur nombre dépend de la traduction choisie. Dans l'exécution committée, la médiane passe de 1,01 ms à 60,10 ms entre 1 000 et 8 000 éléments, pour une pente mesurée de 1,977 ; ces temps varient selon la machine et ne mesurent que le contrôle quadratique.

Les quatre temps mesurés disent la même chose que la pente, mais en la rendant lisible : le passage de `1.01` ms à `60.10` ms quand la taille passe de `1000` à `8000` suit une croissance quasi quadratique, et la pente mesurée est proche de deux sans l'atteindre. L'écart — le facteur `8000^(pente-2)` — est la mesure de ce « presque » : il ne compare que les puissances et ignore les constantes.

Deux limites que la sortie ne franchit pas.

- **La pente est un instantané, pas une asymptotique.** Elle est estimée sur quatre tailles, donc sur trois paliers de doublement ; une constante multiplicative est absorbée dans l'exposant apparent. C'est pourquoi le carnet publie le facteur d'écart au lieu de déclarer la pente égale à deux.
- **Le temps n'est pas le coût.** Le chronomètre mesure une implémentation sur une machine, avec ses tris, ses allocations et ses conversions. Il ne dit rien du nombre d'opérations du modèle, qui est l'objet du carnet. C'est précisément pourquoi l'exercice qui suit impose un **comptage d'itérations** et non une comparaison de temps : l'itération est l'opération du modèle, le temps est celui de l'outillage.

In [9]:
// Ce que compte l'exercice : les iterations cumulees des boucles internes (fixtures 4 -> 3, 7 -> 15).
long? CompterComparaisons(int n) { // TODO : sommer le nombre d'itérations sur n éléments positifs
 return null;
}
var compte=CompterComparaisons(4);
if(compte is null) Console.WriteLine("Exercice 3 à compléter : compter les itérations.");
else { var autre=CompterComparaisons(7); if(compte!=3 || autre!=15) throw new Exception("Comptage incorrect"); Console.WriteLine("Fixtures 4 et 7 validées."); }

Exercice 3 à compléter : compter les itérations.


### Ce que montre cette sortie

La sortie indique que la fonction de comptage reste à écrire, et elle le fait sans erreur — comme les deux autres exercices, le carnet s'exécute de bout en bout.

Ce que cette ligne dit du reste du carnet :

- **Les trois exercices occupent trois granularités.** Un quotient de puissances, un programme qui fabrique une constante, un comptage d'itérations : de l'arithmétique à la traduction en instructions. Aucun n'est décoratif — chacun produit une grandeur que le carnet ne mesure pas autrement.
- **Le carnet mesure, il ne démontre pas.** Ces trois squelettes renvoient une valeur absente là où un calcul est attendu ; tout le reste — le témoin à trois nombres, le contrôle quadratique — a été **exécuté**, et sa sortie est commitée. C'est la distinction que la section qui suit développe.
- **Un exercice se vérifie sans être résolu ici.** Les conditions de contrôle inscrites dans le code disent ce qui sera accepté, et elles sont accessibles au lecteur sans exécution supplémentaire.

Ce que cette sortie **ne** prouve pas : rien sur le coût asymptotique du contrôle. Le comptage d'itérations est un sous-calcul, et il ne faut pas l'assimiler à un nombre de pas de la machine — la traduction choisie change ce nombre, et c'est précisément ce que l'exercice propose de justifier.
## Ce que la source ne prouve pas

Le README exclut notamment les bornes pour entrées réelles et la section 5.1 ; les constantes ne sont pas optimisées. Les hypothèses de difficulté ne sont pas définies dans la formalisation ; les claims énoncent des existences. Le README ne rapporte pas d'exécution de Comparator. La machine est plus faible qu'un word RAM standard par ses opérations limitées, mais autorise une multiplication en un pas et une mémoire adressable par mot. Le programme est fixé avec un nombre $b$ et doit tenir pour toute taille de mot $W \ge b(1+\lfloor\log_2 n\rfloor)$. Quatre gratuités décrites dans MACHINE.md sont : mémoire hors entrée initialisée à zéro, cellules nommées dans le programme hors de portée des mots, choix de la pente $b$ après $\kappa$, et entrées de phases gratuites ; le document précise qu'elles ne changent pas l'exposant.

La série situe cette formalisation hors Mathlib face au manque d'une couche `TIME(f)` discuté dans [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb). [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) examine une autre annonce et [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) replace ces travaux dans le panorama. Le fichier d'énoncé n'importe rien ; cela ne remplace pas une vérification locale des preuves.

## À retenir

Les cinq bornes sont citées, le témoin et le contrôle quadratique sont mesurés. Trois exercices répartis explorent le quotient des exposants, la construction d'une constante et un comptage d'itérations.

## Vérifiez votre compréhension

1. Pourquoi 7 et 6 pas ? 2. Le quotient des puissances prédit-il les temps ? 3. Pourquoi relever à la puissance 1250 ?

<details><summary>Réponses</summary>

1. Le premier chemin atteint `accept` et le second saute plus tôt à `reject`, arrêt inclus. 2. Non : constantes et termes cachés restent inconnus. 3. $1,9992=2499/1250$, ce qui permet d'exprimer la borne avec des puissances entières.
</details>

## Pour aller plus loin

- [EndStatement.lean](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/EndStatement.lean), [README et MACHINE.md](https://github.com/anthropics/formal-math/tree/main/3sum-apsp).
- [03f — Réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) et [07c — Bilan](Complexity-07c-LeBilanMathocalypse-Python.ipynb).

**Navigation** : [Index](README.md) | [Version Python](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb)